<a href="https://colab.research.google.com/github/hussainahmad17/movie-recommender/blob/main/Movie_Recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
import ast # Import the ast module


credits = pd.read_csv('tmdb_5000_credits.csv')
movies = pd.read_csv('tmdb_5000_movies.csv')

# 1 Preprocessing

# merging the data on basis of title and assign it to movies
movies = movies.merge(credits,on="title")

# filtering the columns
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

# check for null values in cloumns
movies.isnull().sum() # means in overview cloumns, we have three empty rows

# drop the null values
movies.dropna(inplace=True)

# Convert string representation of lists to actual lists using ast.literal_eval
columns_to_convert = ['genres', 'keywords', 'cast', 'crew']
for column in columns_to_convert:
    movies[column] = movies[column].apply(ast.literal_eval)

# check for duplicate values in data
# movies.duplicated().sum()

# Now the 'genres' column contains actual lists of dictionaries
def convert(mylist):
  l = []
  for i in mylist: # obj is already a list of dictionaries, so no need for ast.literal_eval
    l.append(i['name'])
  return l

movies["genres"] = movies["genres"].apply(convert)
movies["keywords"] = movies["keywords"].apply(convert)


# get first 3 cast names
def convertcast(mylist):
  l = []
  count = 0
  for i in mylist:
    l.append(i['name'])
    count += 1
    if count == 3:
      break
  return l


movies["cast"] = movies["cast"].apply(convertcast)

# get the name of dict where job is director
def getcrew(mylist):
  l = []
  for i in mylist:
    if i['job'] == 'Director':
      l.append(i['name'])
      break
  return l

movies["crew"] = movies["crew"].apply(getcrew)

# convert the overview into list
movies["overview"] = movies["overview"].apply(lambda x:x.split())

# remove the spaces to avoid the miss match
movies["genres"] = movies["genres"].apply(lambda x:[i.replace(" ","") for i in x])
movies["keywords"] = movies["keywords"].apply(lambda x:[i.replace(" ","") for i in x])
movies["cast"] = movies["cast"].apply(lambda x:[i.replace(" ","") for i in x])
movies["crew"] = movies["crew"].apply(lambda x:[i.replace(" ","") for i in x])

# make a tags columns
movies["tags"] = movies["overview"] + movies["genres"] + movies["keywords"] + movies["cast"] + movies["crew"]

# create new df
new_df = movies[["movie_id","title","tags"]].copy()

# convert the tags arrays into string
new_df["tags"] = new_df["tags"].apply(lambda x:" ".join(x))


new_df.head()

,movie_id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


In [26]:
# now use the nltk lib to convert the words into root words
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

def stem(text):
  y = []
  for i in text.split():
    y.append(ps.stem(i))

  return " ".join(y)


new_df["tags"] = new_df["tags"].apply(stem)

#  now use countvectorizer to convert the text into numerical tokens, and eleminate stop words
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')
vectors = cv.fit_transform(new_df["tags"]).toarray()


# find the cousine simlarity to find the cousine distance
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

# now make a fun to recommend the movies

def recommender(movie):
  # find index of given movie in db
  movie_index = new_df[new_df["title"] == movie].index[0]
  distances = similarity[movie_index]
  movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]
  for i in movies_list:
    print(new_df.iloc[i[0]].title)

recommender("Avatar")


Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [28]:
import pickle
pickle.dump(new_df,open("movies.pkl","wb"))
pickle.dump(similarity,open("similarity.pkl","wb"))